У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

df = pd.read_csv("data/customer_segmentation_train.csv")

print("Розмір датасету:", df.shape)
display(df.head())

print("\nТипи даних:")
display(df.dtypes)

print("\nКількість пропущених значень:")
display(df.isna().sum())

print("\nРозподіл цільової змінної Segment:")
display(df["Segmentation"].value_counts().sort_index())

# ID не є корисною ознакою для моделі
df = df.drop(columns=["ID"])

target_col = "Segmentation"

X = df.drop(columns=[target_col])
y = df[target_col]

num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Числові ознаки:", num_cols)
print("Категоріальні ознаки:", cat_cols)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

# Imputation: fit тільки на train, transform на train і test
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_num = pd.DataFrame(
    num_imputer.fit_transform(X_train_raw[num_cols]),
    columns=num_cols,
    index=X_train_raw.index
)

X_test_num = pd.DataFrame(
    num_imputer.transform(X_test_raw[num_cols]),
    columns=num_cols,
    index=X_test_raw.index
)

X_train_cat = pd.DataFrame(
    cat_imputer.fit_transform(X_train_raw[cat_cols]),
    columns=cat_cols,
    index=X_train_raw.index
)

X_test_cat = pd.DataFrame(
    cat_imputer.transform(X_test_raw[cat_cols]),
    columns=cat_cols,
    index=X_test_raw.index
)

#  Ordinal encoding потрібен для SMOTENC, бо він працює з категоріальними
#  ознаками, але очікує числове представлення категорій.
ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_train_cat_encoded = pd.DataFrame(
    ordinal_encoder.fit_transform(X_train_cat),
    columns=cat_cols,
    index=X_train_raw.index
)

X_test_cat_encoded = pd.DataFrame(
    ordinal_encoder.transform(X_test_cat),
    columns=cat_cols,
    index=X_test_raw.index
)

X_train_prepared = pd.concat([X_train_num, X_train_cat_encoded], axis=1)
X_test_prepared = pd.concat([X_test_num, X_test_cat_encoded], axis=1)

# Масштабування числових ознак: fit тільки на train
scaler = StandardScaler()

X_train_prepared[num_cols] = scaler.fit_transform(X_train_prepared[num_cols])
X_test_prepared[num_cols] = scaler.transform(X_test_prepared[num_cols])

# Для зручності зафіксуємо однаковий порядок колонок
feature_cols = num_cols + cat_cols
X_train_prepared = X_train_prepared[feature_cols]
X_test_prepared = X_test_prepared[feature_cols]

cat_feature_indices = [X_train_prepared.columns.get_loc(col) for col in cat_cols]

print("Train shape:", X_train_prepared.shape)
print("Test shape:", X_test_prepared.shape)
print("Індекси категоріальних ознак для SMOTENC:", cat_feature_indices)

display(X_train_prepared.head())

Розмір датасету: (8068, 11)


,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A



Типи даних:


ID                   int64
Gender              object
Ever_Married        object
Age                  int64
Graduated           object
Profession          object
Work_Experience    float64
Spending_Score      object
Family_Size        float64
Var_1               object
Segmentation        object
dtype: object


Кількість пропущених значень:


ID                   0
Gender               0
Ever_Married       140
Age                  0
Graduated           78
Profession         124
Work_Experience    829
Spending_Score       0
Family_Size        335
Var_1               76
Segmentation         0
dtype: int64


Розподіл цільової змінної Segment:


Segmentation
A    1972
B    1858
C    1970
D    2268
Name: count, dtype: int64

Числові ознаки: ['Age', 'Work_Experience', 'Family_Size']
Категоріальні ознаки: ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']
Train shape: (6454, 9)
Test shape: (1614, 9)
Індекси категоріальних ознак для SMOTENC: [3, 4, 5, 6, 7, 8]


,Age,Work_Experience,Family_Size,Gender,Ever_Married,Graduated,Profession,Spending_Score,Var_1
917,-0.695320,1.970880,-1.231118,0.0,0.0,1.0,0.0,2.0,5.0
3398,1.703982,-0.456381,-0.564314,1.0,1.0,1.0,3.0,0.0,5.0
2045,-0.635337,-0.456381,0.769294,0.0,0.0,1.0,3.0,2.0,5.0
8060,0.264401,-0.759788,2.102901,0.0,1.0,1.0,0.0,0.0,5.0
4604,-0.935250,1.970880,-1.231118,0.0,1.0,0.0,1.0,2.0,6.0


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [7]:
from imblearn.over_sampling import SMOTENC
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import TomekLinks

print("Розподіл класів до ресемплингу:")
display(y_train.value_counts().sort_index())

smote_nc = SMOTENC(
    categorical_features=cat_feature_indices,
    random_state=RANDOM_STATE
)

X_train_smote, y_train_smote = smote_nc.fit_resample(X_train_prepared, y_train)

print("Розподіл класів після SMOTENC:")
display(pd.Series(y_train_smote).value_counts().sort_index())

smote_tomek = SMOTETomek(
    smote=SMOTENC(
        categorical_features=cat_feature_indices,
        random_state=RANDOM_STATE
    ),
    tomek=TomekLinks(),
    random_state=RANDOM_STATE
)

X_train_smote_tomek, y_train_smote_tomek = smote_tomek.fit_resample(
    X_train_prepared,
    y_train
)

print("Розподіл класів після SMOTENC + TomekLinks:")
display(pd.Series(y_train_smote_tomek).value_counts().sort_index())

print("Original train shape:", X_train_prepared.shape)
print("SMOTENC train shape:", X_train_smote.shape)
print("SMOTENC + TomekLinks train shape:", X_train_smote_tomek.shape)

Розподіл класів до ресемплингу:


Segmentation
A    1578
B    1486
C    1576
D    1814
Name: count, dtype: int64

Розподіл класів після SMOTENC:


Segmentation
A    1814
B    1814
C    1814
D    1814
Name: count, dtype: int64

Розподіл класів після SMOTENC + TomekLinks:


Segmentation
A    1814
B    1420
C    1483
D    1484
Name: count, dtype: int64

Original train shape: (6454, 9)
SMOTENC train shape: (7256, 9)
SMOTENC + TomekLinks train shape: (6201, 9)


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score


def train_ovr_logreg_and_evaluate(X_train_data, y_train_data, X_test_data, y_test_data, model_name):
    model = OneVsRestClassifier(
        LogisticRegression(
            max_iter=3000,
            solver="lbfgs",
            random_state=RANDOM_STATE
        )
    )

    model.fit(X_train_data, y_train_data)
    y_pred = model.predict(X_test_data)

    print("=" * 90)
    print(model_name)
    print("=" * 90)
    print(classification_report(y_test_data, y_pred))

    macro_f1 = f1_score(y_test_data, y_pred, average="macro")
    weighted_f1 = f1_score(y_test_data, y_pred, average="weighted")
    accuracy = accuracy_score(y_test_data, y_pred)

    return {
        "model_name": model_name,
        "model": model,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "accuracy": accuracy
    }


results = []

results.append(
    train_ovr_logreg_and_evaluate(
        X_train_prepared,
        y_train,
        X_test_prepared,
        y_test,
        "One-vs-Rest Logistic Regression — original train data"
    )
)

results.append(
    train_ovr_logreg_and_evaluate(
        X_train_smote,
        y_train_smote,
        X_test_prepared,
        y_test,
        "One-vs-Rest Logistic Regression — SMOTENC"
    )
)

results.append(
    train_ovr_logreg_and_evaluate(
        X_train_smote_tomek,
        y_train_smote_tomek,
        X_test_prepared,
        y_test,
        "One-vs-Rest Logistic Regression — SMOTENC + TomekLinks"
    )
)

results_df = pd.DataFrame(results).drop(columns=["model"])
results_df = results_df.sort_values(by="macro_f1", ascending=False)

display(results_df)

best_result = results_df.iloc[0]

print(f"\nНайкраща модель: {best_result['model_name']}")
print(f"Macro F1-score найкращої моделі: {best_result['macro_f1']:.4f}")

One-vs-Rest Logistic Regression — original train data
              precision    recall  f1-score   support

           A       0.39      0.39      0.39       394
           B       0.41      0.08      0.14       372
           C       0.47      0.64      0.54       394
           D       0.59      0.79      0.67       454

    accuracy                           0.49      1614
   macro avg       0.46      0.48      0.44      1614
weighted avg       0.47      0.49      0.45      1614

One-vs-Rest Logistic Regression — SMOTENC
              precision    recall  f1-score   support

           A       0.41      0.39      0.40       394
           B       0.37      0.15      0.21       372
           C       0.48      0.63      0.54       394
           D       0.62      0.78      0.69       454

    accuracy                           0.50      1614
   macro avg       0.47      0.49      0.46      1614
weighted avg       0.48      0.50      0.47      1614

One-vs-Rest Logistic Regression — 

,model_name,macro_f1,weighted_f1,accuracy
1,One-vs-Rest Logistic Regression — SMOTENC,0.460606,0.472429,0.501239
2,One-vs-Rest Logistic Regression — SMOTENC + To...,0.438500,0.453326,0.504337
0,One-vs-Rest Logistic Regression — original tra...,0.436583,0.449456,0.493185



Найкраща модель: One-vs-Rest Logistic Regression — SMOTENC
Macro F1-score найкращої моделі: 0.4606


**Висновки:**
Обрана метрика для порівняння моделей - macro F1-score.
Macro F1-score добре підходить для цієї задачі, тому що класи незбалансовані, а ця метрика однаково враховує якість класифікації для кожного класу. Між моделями є помітна різниця. Найкращою є модель One-vs-Rest Logistic Regression — SMOTENC з найбільшим macro F1-score (0.4606), оскільки вона краще балансує якість по всіх сегментах клієнтів.